# Experiment 7: Parsing Using Context Free Grammar (CFG)

**Mapped CO:** CO4  
**BT Level:** BT4

## Aim
Define a Context Free Grammar and parse sentences into constituent structures using NLTK.

## Objectives
- Understand terminals, non-terminals, productions, and the start symbol.
- Construct a small CFG for English sentences.
- Generate parse trees using a chart parser.
- Analyze grammatical ambiguity through multiple parses.

## Theory
A Context Free Grammar (CFG) describes how non-terminal symbols expand into sequences of terminals and non-terminals. In constituency parsing, a sentence is analyzed into phrases such as noun phrases (NP), verb phrases (VP), and prepositional phrases (PP). A chart parser reuses partial results while searching for a derivation from the start symbol that covers the complete input sentence.

Natural language can be structurally ambiguous. For example, in `I saw the dog with a telescope`, the PP `with a telescope` may modify the seeing event or the dog, producing more than one valid parse when the grammar permits both attachments.

## Methodology
1. Define lexical and phrase-structure CFG rules.
2. Create an NLTK chart parser from the grammar.
3. Tokenize sentences covered by the grammar vocabulary.
4. Generate and print all valid parse trees.
5. Count parses and inspect structural ambiguity.
6. Test a sentence that is not covered by the grammar.

This controlled, hand-written dataset demonstrates parsing mechanics without downloading any NLTK corpora.

In [1]:
import nltk

print(f"NLTK version: {nltk.__version__}")

NLTK version: 3.10.4


## 1. Define the CFG

The grammar has lexical rules for the controlled vocabulary and phrase-structure rules for sentences, noun phrases, verb phrases, and prepositional phrases. The two VP rules allow a PP to attach to either the VP or the NP, making the example sentence ambiguous.

In [2]:
grammar = nltk.CFG.fromstring("""
    S -> NP VP
    NP -> PRON | DET N | DET N PP | NP PP
    VP -> V NP | V NP PP | VP PP
    PP -> P NP

    PRON -> 'I'
    DET -> 'the' | 'a'
    N -> 'dog' | 'cat' | 'telescope'
    V -> 'saw' | 'chased'
    P -> 'with'
""")

# CFG does not provide a nonterminals() method in this NLTK version.
# Extract non-terminals from the left-hand side of every production.
productions = grammar.productions()
non_terminals = sorted({
    str(production.lhs())
    for production in productions
})

print("Start symbol:", grammar.start())
print("Non-terminals:", non_terminals)
print("Number of productions:", len(productions))

print("\nProductions:")
for production in productions:
    print(" ", production)

Start symbol: S
Non-terminals: ['DET', 'N', 'NP', 'P', 'PP', 'PRON', 'S', 'V', 'VP']
Number of productions: 18

Productions:
  S -> NP VP
  NP -> PRON
  NP -> DET N
  NP -> DET N PP
  NP -> NP PP
  VP -> V NP
  VP -> V NP PP
  VP -> VP PP
  PP -> P NP
  PRON -> 'I'
  DET -> 'the'
  DET -> 'a'
  N -> 'dog'
  N -> 'cat'
  N -> 'telescope'
  V -> 'saw'
  V -> 'chased'
  P -> 'with'


In [3]:
parser = nltk.ChartParser(grammar)

def parse_sentence(sentence):
    """Return all CFG parses for a whitespace-tokenized sentence."""
    tokens = sentence.split()
    return list(parser.parse(tokens))

def show_parses(sentence):
    """Print every parse tree and return the list of trees."""
    trees = parse_sentence(sentence)
    print(f"Sentence: {sentence}")
    print(f"Number of parses: {len(trees)}")
    for number, tree in enumerate(trees, start=1):
        print(f"\nParse {number}:")
        print(tree)
    return trees

## 2. Parse an unambiguous sentence

The sentence below is covered by the grammar and has a single constituent structure.

In [4]:
simple_trees = show_parses("the dog chased a cat")
assert len(simple_trees) == 1, "The controlled simple sentence should have one parse."

Sentence: the dog chased a cat
Number of parses: 1

Parse 1:
(S (NP (DET the) (N dog)) (VP (V chased) (NP (DET a) (N cat))))


## 3. Observe PP-attachment ambiguity

The phrase `with a telescope` can attach to the VP (the seeing was done with a telescope) or to the NP `the dog` (the dog with a telescope was seen). The grammar therefore produces multiple valid trees.

In [5]:
ambiguous_sentence = "I saw the dog with a telescope"
ambiguous_trees = show_parses(ambiguous_sentence)

print("\nAmbiguity detected:", len(ambiguous_trees) > 1)
assert len(ambiguous_trees) > 1, "The ambiguity example should have multiple parses."

Sentence: I saw the dog with a telescope
Number of parses: 4

Parse 1:
(S
  (NP (PRON I))
  (VP
    (VP (V saw) (NP (DET the) (N dog)))
    (PP (P with) (NP (DET a) (N telescope)))))

Parse 2:
(S
  (NP (PRON I))
  (VP
    (V saw)
    (NP (DET the) (N dog))
    (PP (P with) (NP (DET a) (N telescope)))))

Parse 3:
(S
  (NP (PRON I))
  (VP
    (V saw)
    (NP
      (NP (DET the) (N dog))
      (PP (P with) (NP (DET a) (N telescope))))))

Parse 4:
(S
  (NP (PRON I))
  (VP
    (V saw)
    (NP (DET the) (N dog) (PP (P with) (NP (DET a) (N telescope))))))

Ambiguity detected: True


In [6]:
# Text rendering is reliable in every notebook environment. Drawing is optional
# because graphical windows may not be available in a remote or headless session.
import os

graphics_available = os.name != "posix" or bool(os.environ.get("DISPLAY"))
# Text output works in every notebook environment.
# Graphical tree drawing is attempted only when Tkinter is available.

if not ambiguous_trees:
    print("No parse tree is available to draw.")
else:
    try:
        import tkinter
        import nltk.draw.tree

        ambiguous_trees[0].draw()

    except (ImportError, OSError, RuntimeError) as error:
        print("Tree drawing skipped because graphical Tkinter support is unavailable.")
        print(f"Details: {error}")

Tree drawing skipped because graphical Tkinter support is unavailable.
Details: libtk8.6.so: cannot open shared object file: No such file or directory


## 4. Failure case

A chart parser returns no trees when the sentence contains words or structures not licensed by the grammar. This is a grammatical coverage failure, not a Python error.

In [7]:
uncovered_sentence = "birds fly"

try:
    failed_trees = parse_sentence(uncovered_sentence)

    if failed_trees:
        print(f"Unexpectedly found {len(failed_trees)} parse(s).")
    else:
        print(f"No valid parse found for: {uncovered_sentence!r}")
        print("Reason: the sentence structure is not licensed by the CFG.")

except ValueError as error:
    failed_trees = []

    print(f"No parse found for: {uncovered_sentence!r}")
    print("Reason: the grammar does not cover one or more input words.")
    print(f"Parser message: {error}")

assert not failed_trees

No parse found for: 'birds fly'
Reason: the grammar does not cover one or more input words.
Parser message: Grammar does not cover some of the input words: "'birds', 'fly'".


## Result and outcome

- The CFG identifies terminals, non-terminals, productions, and the start symbol.
- `nltk.ChartParser` generates constituency trees for covered sentences.
- The sentence `I saw the dog with a telescope` demonstrates structural ambiguity through multiple parses.
- An uncovered sentence produces zero valid trees, showing the limits of the controlled grammar.

## References
- [NLTK parsing guide](https://www.nltk.org/howto/parse.html)
- [NLTK Book, Chapter 8: Analyzing Sentence Structure](https://www.nltk.org/book/ch08.html)
- [Speech and Language Processing](https://web.stanford.edu/~jurafsky/slp3/)
- [Demonstrative video search](https://www.youtube.com/results?search_query=NPTEL+context+free+grammar+constituency+parsing+NLP)